# ColabDFT MO — molecular orbital (HOMO, LUMO) view

Runs a DFT single point with [GPU4PySCF](https://github.com/pyscf/gpu4pyscf) on the input structure as it is, and shows **molecular orbitals such as HOMO and LUMO as isosurfaces in 3D**. The structure is not optimized. No coding needed. UMA is not used, so no Hugging Face token is needed.

## How to use

1. In **Runtime → Change runtime type**, choose **T4 GPU**
2. In **1. Input form** below, set the output name, charge, spin multiplicity, functional, basis set and the orbitals to show
3. Choose **Runtime → Run all**
4. A file selection button appears in **2. File selection and checks**. Choose an xyz or a Gaussian gjf / com file. With the input set to draw, a structure editor appears instead: draw the molecule and press "Use this structure". You can leave it running after that
5. Check the 3D view of the orbitals and their energies in the cells below (click the energy level diagram to see other orbitals), and download the ZIP of results in the last cell

Only the structure is read from a gjf; the method, charge and multiplicity come from the form. If an xyz holds several structures, only the first is computed.

To run again, repeat steps 2–5 (the install is skipped). In the file selection, Cancel upload reuses the previous file. With draw, the previous drawing is shown first.

## Output (contents of the ZIP)

| File | Contents |
|---|---|
| `<output name>_spe.xyz` | The computed structure. Energy and settings in the comment line |
| `<output name>_mo<number>_<orbital>.cube` | Orbital values on the grid (Bohr^-3/2). For example `_mo5_homo.cube`, `_mo6_lumo.cube`, `_mo4_homom1.cube` (HOMO-1). Open shells get `_alpha` / `_beta`. Opens in GaussView and similar programs |
| `<output name>_mo_energies.csv` | Energies (Hartree and eV) and occupations of all orbitals |
| `<output name>_dft.csv` | DFT single point energy |
| `<output name>_log.txt` | Input settings, software versions, results, HOMO and LUMO energies and the gap |

## Notes

- Isosurfaces are **blue for positive and red for negative**. The overall sign (phase) of an orbital has no meaning, so blue and red can swap between runs.
- The orbitals are Kohn–Sham orbitals. Orbital energies and the HOMO-LUMO gap depend strongly on the functional (they do not match experimental ionization energies or absorption wavelengths).
- Give the orbitals to show separated by commas, such as "HOMO, LUMO". HOMO-1, LUMO+1 and orbital numbers (from 1, lowest energy first) also work.
- For open shells (multiplicity 2 or more), α and β orbitals are shown separately. The α HOMO is the SOMO.
- In "7. Show the orbitals", clicking a level in the energy level diagram computes and shows that orbital. 8 levels below HOMO and above LUMO are listed (core orbitals are left out). The orbitals given in the form are saved in the notebook; the others can be computed only while the runtime is connected.
- Only the orbitals given in the form go into the ZIP as cube files. To save an orbital found in the diagram, add it to the form and run again.
- The 3D view uses a thinned grid so that the notebook stays light. The cube files in the ZIP are not thinned.
- For elements with an ECP (Rb and heavier), the core electrons are replaced by the ECP, so the orbital numbers are smaller by that many (HOMO and LUMO work as usual).
- DFT runs on GPU4PySCF (density fitting). The T4 is slow at double precision, so large molecules and def2-TZVPD take time. Without a GPU, DFT runs on the CPU with PySCF.
- Closed shells (multiplicity 1) use RKS, open shells UKS.
- With draw, hydrogens are added to the drawing and RDKit makes the 3D structure (10 ETKDG conformers optimized with MMFF94, lowest one kept). The structure is not optimized further; for an accurate structure, use an xyz optimized with [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt_en.ipynb). The charge comes from the form (a note is shown if it differs from the formal charge of the drawing).
- Supported elements are H–Rn (Z = 1–86). Periodic systems (crystals, surfaces) are not supported.
- To optimize the structure with UMA before the single point, use [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt_en.ipynb).
- To see the electrostatic potential, use [ColabDFT_ESP](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabDFT_ESP_en.ipynb).
- If you only need the energy, [ColabDFT_SPE](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabDFT_SPE_en.ipynb) finishes sooner.

## License

- This notebook and the shared code (`colabuma`): [MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- Libraries installed at runtime are used under their own licenses. See [License in the README](https://github.com/RChibaChiba/ColabChem/blob/main/README.md#license)

## Citation

If you use this notebook in a publication, please cite:

- PySCF (Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020))
- GPU4PySCF (X. Wu *et al.*, arXiv:2404.09452 (2024))

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
#@markdown file: choose an xyz or gjf file. draw: draw the structure in the editor shown in the next cell (hydrogens are added and RDKit makes it 3D).
input_mode = "file"  #@param ["file", "draw"]
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### DFT (gpu4pyscf)
#@markdown ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]

#@markdown ### Orbital view
#@markdown Orbitals to show, separated by commas. For example `HOMO, LUMO` or `HOMO-1, HOMO, LUMO, LUMO+1`. Orbital numbers (from 1) also work.
orbitals = "HOMO, LUMO"  #@param {type:"string"}
#@markdown Isovalue ± (Bohr^-3/2). 0.02 is the GaussView default. A larger value gives smaller surfaces.
isovalue = 0.02  #@param {type:"number"}
#@markdown Grid quality. A finer grid looks better but takes longer and makes larger files.
grid_quality = "standard"  #@param ["coarse", "standard", "fine"]

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection (or drawing the structure) happens here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, mo, output, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

structure.check_multiplicity(multiplicity)
functional, basis = dft.parse_method(dft_functional, dft_basis)
orbital_specs = mo.check_form(orbitals, isovalue, grid_quality)
job = dft.Job(structure.out_prefix(output_name), charge, multiplicity, functional, basis)
env.check_gpu()
upload_name, upload_data = structure.get_input(input_mode)
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install) { display-mode: "form" }
DEVICE = env.setup_dft(drawing=structure.is_drawing(input_mode))

In [ ]:
#@title 4. Prepare the structure { display-mode: "form" }
atoms, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity, uma=False)
structure.show_numbered(atoms)
ok(f"Structure ready: {atoms.get_chemical_formula()} ({len(atoms)} atoms).")

In [ ]:
#@title 5. DFT { display-mode: "form" }
dft_res = dft.run([("Input structure", atoms)], charge, multiplicity, functional, basis, keep_orbitals=True)
files, dft_log = dft.report(dft_res, None, geometry="input structure")
_p = dft_res["points"][0]
files["spe.xyz"] = structure.xyz_block(atoms.get_chemical_symbols(), atoms.positions,
                                       f"energy_Ha={_p['energy']:.10f} converged={_p['converged']} "
                                       f"method={functional}/{basis} charge={charge} multiplicity={multiplicity}")

In [ ]:
#@title 6. Compute the orbitals { display-mode: "form" }
mo_res = mo.compute(dft_res["points"][0], atoms, orbital_specs, isovalue, grid_quality)

In [ ]:
#@title 7. Show the orbitals { display-mode: "form" }
#@markdown Click a level in the energy level diagram on the left to show that orbital on the right (8 levels below HOMO and above LUMO). Blue is positive and red is negative. Rotate and zoom with the mouse.
_f, mo_log = mo.report(mo_res)
files.update(_f)

In [ ]:
#@title 8. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabDFT MO log", source_desc, job, [dft_log, mo_log])
zip_path = output.save_zip(job.out_prefix, files)